# 09 SVC With Basic Image Features

This notebook trains a basic `SVC`, then tunes only SVC using grouped cross-validation on the training split. SVC tries to find a decision boundary with a maximum margin in scaled feature space.

## 1. Project setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/jp/MSE446_Nanoparticle_Ordering')

## 2. Imports

In [2]:
from src.extract_basic_features import BASIC_FEATURE_COLUMNS
from src.train_logistic_regression import FEATURES_CSV, load_features
from src.train_svc import (
    BASIC_SCORES_CSV,
    TUNED_SCORES_CSV,
    BASIC_CONFUSION_MATRIX_FIGURE,
    TUNED_CONFUSION_MATRIX_FIGURE,
    build_comparison_table,
    print_overfitting_summary,
    train_svc,
)

## 3. Paths and configuration

In [3]:
print(f"Basic features CSV: {FEATURES_CSV}")
print(f"Basic scores CSV: {BASIC_SCORES_CSV}")
print(f"Tuned scores CSV: {TUNED_SCORES_CSV}")
print(f"Basic confusion matrix figure: {BASIC_CONFUSION_MATRIX_FIGURE}")
print(f"Tuned confusion matrix figure: {TUNED_CONFUSION_MATRIX_FIGURE}")

Basic features CSV: /home/jp/MSE446_Nanoparticle_Ordering/data/features_basic.csv
Basic scores CSV: /home/jp/MSE446_Nanoparticle_Ordering/results/model_scores_svc_basic.csv
Tuned scores CSV: /home/jp/MSE446_Nanoparticle_Ordering/results/model_scores_svc_tuned.csv
Basic confusion matrix figure: /home/jp/MSE446_Nanoparticle_Ordering/results/figures/confusion_matrix_svc_basic.png
Tuned confusion matrix figure: /home/jp/MSE446_Nanoparticle_Ordering/results/figures/confusion_matrix_svc_tuned.png


## 4. Load metadata

In [4]:
features = load_features(FEATURES_CSV)
features.head()

,filename,label,sample,area,area_group,kv,mm,mag,param_group,mean_intensity,...,min_intensity,max_intensity,p10_intensity,p25_intensity,p50_intensity,p75_intensity,p90_intensity,entropy,bright_pixel_ratio,edge_density
0,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.354890,...,0.160784,0.964706,0.266667,0.298039,0.345098,0.396078,0.447059,6.260059,0.099014,0.161224
1,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.414923,...,0.094118,1.000000,0.329412,0.376471,0.407843,0.447059,0.486275,6.191408,0.098190,0.113770
2,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.311404,...,0.141176,1.000000,0.247059,0.266667,0.286275,0.317647,0.423529,5.814114,0.097717,0.160583
3,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.326249,...,0.113725,1.000000,0.235294,0.266667,0.313726,0.368627,0.419608,6.261398,0.098694,0.164108
4,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.260067,...,0.000000,1.000000,0.117647,0.176471,0.235294,0.294118,0.415686,6.846660,0.099258,0.171616


## 5. Sanity checks

In [5]:
print(f"Rows: {len(features)}")
print("Label counts:")
print(features["label"].value_counts().sort_index())
print(f"Image feature columns: {len(BASIC_FEATURE_COLUMNS)}")
print(BASIC_FEATURE_COLUMNS)
assert set(BASIC_FEATURE_COLUMNS).issubset(features.columns)
assert features["area_group"].notna().all()

Rows: 1000
Label counts:
label
disordered    250
ordered       750
Name: count, dtype: int64
Image feature columns: 12
['mean_intensity', 'std_intensity', 'min_intensity', 'max_intensity', 'p10_intensity', 'p25_intensity', 'p50_intensity', 'p75_intensity', 'p90_intensity', 'entropy', 'bright_pixel_ratio', 'edge_density']


## 6. Main analysis

Only image-derived feature columns are used as predictors. Metadata columns are used for labels, auditing, and grouped splitting, not as model inputs. SVC depends on margins and kernel distances, so `StandardScaler` is included inside the model pipeline.

In [6]:
(
    basic_scores,
    basic_report,
    basic_matrix,
    basic_figure_path,
    tuned_scores,
    tuned_report,
    tuned_matrix,
    tuned_figure_path,
    y_train,
    y_test,
    search,
) = train_svc(features)

basic_scores

,model,train_accuracy,test_accuracy,train_balanced_accuracy,test_balanced_accuracy,train_macro_precision,macro_precision,train_macro_recall,macro_recall,train_macro_f1,...,selected_split_seed,split_distribution_distance,best_cv_macro_f1,best_params,C,kernel,gamma,class_weight,train_size,test_size
0,svc_basic,0.902582,0.885135,0.869327,0.86036,0.870489,0.842593,0.869327,0.86036,0.869906,...,140,0.0,None,,1.0,rbf,scale,balanced,852,148


## 7. Results/figures

Balanced accuracy and macro F1 matter more than raw accuracy because the dataset has many more ordered images than disordered images. Macro F1 is also used for tuning.

In [7]:
basic_scores.to_csv(BASIC_SCORES_CSV, index=False)
tuned_scores.to_csv(TUNED_SCORES_CSV, index=False)

print("Basic SVC per-class precision/recall/F1:")
print(basic_report)
print("Tuned SVC per-class precision/recall/F1:")
print(tuned_report)

print_overfitting_summary(basic_scores, "Basic SVC")
print_overfitting_summary(tuned_scores, "Tuned SVC")

tuned_row = tuned_scores.iloc[0]
print("\nTuning summary:")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV macro F1: {search.best_score_:.4f}")
print(f"Final test macro F1: {tuned_row['macro_f1']:.4f}")
print(f"Final test balanced accuracy: {tuned_row['test_balanced_accuracy']:.4f}")
print(f"Disordered recall: {tuned_row['disordered_recall']:.4f}")

print("\nModel comparison:")
print(build_comparison_table(basic_scores, tuned_scores))

print(f"\nSaved basic scores to {BASIC_SCORES_CSV}")
print(f"Saved tuned scores to {TUNED_SCORES_CSV}")
print(f"Saved basic confusion matrix figure to {basic_figure_path}")
print(f"Saved tuned confusion matrix figure to {tuned_figure_path}")

tuned_scores

Basic SVC per-class precision/recall/F1:
              precision    recall  f1-score   support

  disordered       0.75      0.81      0.78        37
     ordered       0.94      0.91      0.92       111

    accuracy                           0.89       148
   macro avg       0.84      0.86      0.85       148
weighted avg       0.89      0.89      0.89       148

Tuned SVC per-class precision/recall/F1:
              precision    recall  f1-score   support

  disordered       0.82      0.84      0.83        37
     ordered       0.95      0.94      0.94       111

    accuracy                           0.91       148
   macro avg       0.88      0.89      0.88       148
weighted avg       0.91      0.91      0.91       148


Basic SVC train vs test:
  train accuracy:           0.9026
  test accuracy:            0.8851
  train balanced accuracy:  0.8693
  test balanced accuracy:   0.8604
  train macro F1:           0.8699
  test macro F1:            0.8508

Tuned SVC train vs test:
  

,model,train_accuracy,test_accuracy,train_balanced_accuracy,test_balanced_accuracy,train_macro_precision,macro_precision,train_macro_recall,macro_recall,train_macro_f1,...,selected_split_seed,split_distribution_distance,best_cv_macro_f1,best_params,C,kernel,gamma,class_weight,train_size,test_size
0,svc_tuned,0.899061,0.912162,0.868545,0.887387,0.86403,0.880622,0.868545,0.887387,0.866248,...,140,0.0,0.765351,"{'svc__C': 100, 'svc__class_weight': 'balanced...",100,rbf,0.01,balanced,852,148


## 8. Notes for report

- SVC tries to find a decision boundary with a maximum margin.
- A linear kernel tests whether the classes separate with a linear boundary.
- An RBF kernel allows nonlinear class boundaries.
- Scaling is required because SVC depends on distances and margins in feature space.
- Macro F1 is used for tuning because the dataset is imbalanced.
- This step does not add HOG, graph features, augmentation, or CNNs.